
# Curved multi-branch toy V2 evaluation

V2 focuses on fair paired-batch comparison.

Main fixes:
1. **Without projection / with projection use the exact same sampled batch**.
2. Default projection strength is reduced to a conservative range.
3. Supports the stronger BP-GI2SB V2 model where transport and correction heads also see backbone feature $h$.
4. All metrics are computed on matched samples.


In [ ]:

from pathlib import Path
import os
import sys
import math
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "code").is_dir():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
CODE_DIR = PACKAGE_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from train_curved_multibranch_toy_v2 import (
    BPModelV2,
    SingleHeadModel,
    sample_batch,
    sample_reverse,
    make_branch_curves,
    union_bridge_curve,
    nearest_distance_to_points,
    branch_accuracy,
    toy_std,
    to_np,
)

plt.rcParams["figure.dpi"] = 300


## 1. Configuration

In [ ]:

# Set CASE2_TRIAD_CHECKPOINT and CASE2_SINGLE_CHECKPOINT when running locally.
def configured_checkpoint(env_name, relative_name):
    value = os.environ.get(env_name, relative_name)
    path = Path(value)
    return path if path.is_absolute() else PACKAGE_ROOT / path

BP_CKPT = configured_checkpoint("CASE2_TRIAD_CHECKPOINT", "checkpoints/triad_latest.pt")
SINGLE_CKPT = configured_checkpoint("CASE2_SINGLE_CHECKPOINT", "checkpoints/single_latest.pt")

OUTDIR = Path("eval_toy_curved_multibranch_v2")
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use the same batch for fair paired comparisons.
N_SAMPLES = 4096
N_CURVE = 2500

# Main sampling setup.
NFE_MAIN = 20
RHO_MAIN = 0.02
RHO_SCHEDULE_MAIN = "sin2"

# Scans.
NFE_LIST = [1, 2, 5, 10, 20, 40, 60]
RHO_LIST = [0.0, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
RHO_SCHEDULES = ["constant", "sin2", "poly"]
KAPPA_LIST = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

print("Device:", DEVICE)
print("Output:", OUTDIR)


## 2. Load models

In [ ]:

def dict_to_namespace(d):
    class Obj:
        pass
    obj = Obj()
    for k, v in d.items():
        setattr(obj, k, v)
    return obj

def load_model(ckpt_path):
    ckpt_path = Path(ckpt_path)
    payload = torch.load(ckpt_path, map_location=DEVICE)
    args = dict_to_namespace(payload["args"])

    if args.model_type == "bp":
        detach_flag = getattr(args, "detach_m_for_heads", False)
        model = BPModelV2(
            hidden=args.hidden,
            depth=args.depth,
            gamma_max=args.gamma_max,
            detach_m_for_heads=detach_flag,
        )
    elif args.model_type == "single":
        model = SingleHeadModel(hidden=args.hidden, depth=args.depth)
    else:
        raise ValueError(args.model_type)

    model.load_state_dict(payload["model"], strict=True)
    model.to(DEVICE).eval()
    return model, args

bp_model, bp_args = load_model(BP_CKPT)
print("Loaded BP model:", BP_CKPT)
print("BP kappa:", bp_args.kappa)

single_model, single_args = None, None
if Path(SINGLE_CKPT).exists():
    single_model, single_args = load_model(SINGLE_CKPT)
    print("Loaded single-head model:", SINGLE_CKPT)
else:
    print("Single-head checkpoint not found. Single-head comparisons will be skipped.")


## 3. Metric functions

In [ ]:

@torch.no_grad()
def make_eval_args(args, kappa_override=None):
    args_eval = copy.copy(args)
    if kappa_override is not None:
        args_eval.kappa = float(kappa_override)
    return args_eval

@torch.no_grad()
def make_paired_batch(args, n_samples=N_SAMPLES, kappa_override=None, seed=12345):
    # Fixed seed for deterministic paired comparison, without altering global randomness too much.
    state_cpu = torch.random.get_rng_state()
    state_cuda = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    args_eval = make_eval_args(args, kappa_override)
    batch = sample_batch(n_samples, DEVICE, args_eval)

    torch.random.set_rng_state(state_cpu)
    if torch.cuda.is_available() and state_cuda is not None:
        torch.cuda.set_rng_state_all(state_cuda)

    return batch, args_eval

@torch.no_grad()
def nearest_distance_to_bridge_union(points, t_value, args_eval):
    curve = union_bridge_curve(N_CURVE, DEVICE, kappa=args_eval.kappa, t_value=float(t_value))
    return nearest_distance_to_points(points, curve)

@torch.no_grad()
def endpoint_rmse(final, x0):
    return torch.sqrt(((final - x0).pow(2).sum(dim=1)).mean()).item()

@torch.no_grad()
def final_dist_to_clean_union(final, args_eval):
    curves = make_branch_curves(N_CURVE, DEVICE, kappa=args_eval.kappa, t_value=0.0)
    ref = torch.cat([curves[-1.0]["x0"], curves[1.0]["x0"]], dim=0)
    return nearest_distance_to_points(final, ref).mean().item()

@torch.no_grad()
def trajectory_metrics(traj):
    diffs = traj[1:] - traj[:-1]
    step_len = torch.linalg.norm(diffs, dim=-1)
    path_len = step_len.sum(dim=0)
    chord = torch.linalg.norm(traj[-1] - traj[0], dim=-1)
    eps = 1e-8

    out = {
        "path_length": path_len.mean().item(),
        "excess_path_length": (path_len - chord).mean().item(),
        "path_tortuosity": (path_len / (chord + eps)).mean().item(),
        "path_efficiency": (chord / (path_len + eps)).mean().item(),
    }

    if traj.shape[0] >= 3:
        v1 = diffs[:-1]
        v2 = diffs[1:]
        dot = (v1 * v2).sum(dim=-1)
        denom = torch.linalg.norm(v1, dim=-1) * torch.linalg.norm(v2, dim=-1) + eps
        ang = torch.arccos(torch.clamp(dot / denom, -1.0, 1.0)) * 180.0 / math.pi
        out["total_turning_angle_deg"] = ang.sum(dim=0).mean().item()
        out["mean_turning_angle_deg"] = ang.mean(dim=0).mean().item()
    else:
        out["total_turning_angle_deg"] = 0.0
        out["mean_turning_angle_deg"] = 0.0

    return out

@torch.no_grad()
def trajectory_dist_to_bridge(traj, times, args_eval):
    ds = []
    for j in range(traj.shape[0]):
        d = nearest_distance_to_bridge_union(traj[j], float(times[j]), args_eval)
        ds.append(d.mean())
    return torch.stack(ds).mean().item()

@torch.no_grad()
def evaluate_on_batch(
    model,
    args,
    batch,
    args_eval,
    nfe=NFE_MAIN,
    rho_max=RHO_MAIN,
    projection_mode="adaptive",
    rho_sched=RHO_SCHEDULE_MAIN,
):
    x1 = batch["x1"]
    x0 = batch["x0"]
    y = batch["y"]

    # Use model-specific args for model_type and schedule hyperparameters, but evaluate kappa from args_eval.
    run_args = copy.copy(args)
    run_args.kappa = args_eval.kappa

    final, traj, times = sample_reverse(
        model,
        x1,
        run_args,
        nfe=nfe,
        rho_max=rho_max,
        rho_sched=rho_sched,
        projection_mode=projection_mode,
        return_traj=True,
    )

    row = {
        "model_type": args.model_type,
        "kappa_eval": args_eval.kappa,
        "nfe": nfe,
        "rho_max": rho_max,
        "rho_sched": rho_sched,
        "projection_mode": projection_mode,
        "endpoint_rmse": endpoint_rmse(final, x0),
        "final_dist_to_M0": final_dist_to_clean_union(final, args_eval),
        "trajectory_dist_to_Mt": trajectory_dist_to_bridge(traj, times, args_eval),
        "branch_acc": branch_accuracy(final, y, N_CURVE, DEVICE, args_eval.kappa),
    }
    row.update(trajectory_metrics(traj))
    return row, {"batch": batch, "final": final, "traj": traj, "times": times, "args_eval": args_eval}

@torch.no_grad()
def bridge_diagnostic(model, args, n_samples=N_SAMPLES, n_t_bins=5, kappa_override=None, seed=6789):
    if args.model_type != "bp":
        return pd.DataFrame()

    batch, args_eval = make_paired_batch(args, n_samples=n_samples, kappa_override=kappa_override, seed=seed)
    xt, t, x1, s = batch["xt"], batch["t"], batch["x1"], batch["s"]
    x0, mt = batch["x0"], batch["mt"]

    out = model(xt, t, x1, s)
    o_target = (xt - x0) / s

    rows = []
    edges = torch.linspace(0.0, 1.0, n_t_bins + 1, device=DEVICE)
    for i in range(n_t_bins):
        lo, hi = edges[i], edges[i+1]
        if i == n_t_bins - 1:
            mask = (t[:, 0] >= lo) & (t[:, 0] <= hi)
        else:
            mask = (t[:, 0] >= lo) & (t[:, 0] < hi)
        if mask.sum() == 0:
            continue
        rows.append({
            "t_bin": f"{float(lo):.2f}-{float(hi):.2f}",
            "n": int(mask.sum().item()),
            "bridge_mse": F.mse_loss(out["m"][mask], mt[mask]).item(),
            "endpoint_mse": F.mse_loss(out["x0_hat"][mask], x0[mask]).item(),
            "target_mse": F.mse_loss(out["o_hat"][mask], o_target[mask]).item(),
            "corr_energy": out["c"][mask].pow(2).mean().item(),
        })
    return pd.DataFrame(rows)


## 4. Direct paired comparison

In [ ]:

# One fixed batch for all direct-comparison rows.
direct_batch, direct_args_eval = make_paired_batch(bp_args, n_samples=N_SAMPLES, kappa_override=bp_args.kappa, seed=2026)

rows = []

row, cache_no = evaluate_on_batch(
    bp_model, bp_args, direct_batch, direct_args_eval,
    nfe=NFE_MAIN,
    rho_max=0.0,
    projection_mode="none",
    rho_sched=RHO_SCHEDULE_MAIN,
)
row["mode_name"] = "BP-no-projection"
rows.append(row)

row, cache_proj = evaluate_on_batch(
    bp_model, bp_args, direct_batch, direct_args_eval,
    nfe=NFE_MAIN,
    rho_max=RHO_MAIN,
    projection_mode="adaptive",
    rho_sched=RHO_SCHEDULE_MAIN,
)
row["mode_name"] = "BP-projection"
rows.append(row)

if single_model is not None:
    row, cache_single = evaluate_on_batch(
        single_model, single_args, direct_batch, direct_args_eval,
        nfe=NFE_MAIN,
        rho_max=0.0,
        projection_mode="none",
        rho_sched=RHO_SCHEDULE_MAIN,
    )
    row["mode_name"] = "Single-head"
    rows.append(row)

df_direct = pd.DataFrame(rows)
df_direct.to_csv(OUTDIR / "direct_comparison_metrics_v2.csv", index=False)
df_direct


In [ ]:

def barplot(df, metric, title=None, fname=None):
    labels = df["mode_name"].values
    vals = df[metric].values
    plt.figure(figsize=(7,4))
    plt.bar(labels, vals)
    plt.xticks(rotation=20, ha="right")
    plt.ylabel(metric)
    plt.title(title or metric)
    for i, v in enumerate(vals):
        label = f"{v:.3e}" if abs(v) < 0.01 else f"{v:.4f}"
        plt.text(i, v, label, ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    if fname:
        plt.savefig(OUTDIR / fname, dpi=180)
    plt.show()

for metric in ["endpoint_rmse", "trajectory_dist_to_Mt", "branch_acc", "excess_path_length", "path_tortuosity", "total_turning_angle_deg"]:
    barplot(df_direct, metric, fname=f"direct_v2_{metric}.png")


## 5. Bridge diagnostic by t-bin

In [ ]:

df_bridge = bridge_diagnostic(bp_model, bp_args, n_samples=N_SAMPLES, n_t_bins=5, kappa_override=bp_args.kappa)
df_bridge.to_csv(OUTDIR / "bridge_diagnostic_by_tbin_v2.csv", index=False)
df_bridge


In [ ]:

if len(df_bridge):
    for metric in ["bridge_mse", "endpoint_mse", "target_mse", "corr_energy"]:
        plt.figure(figsize=(6,4))
        plt.plot(df_bridge["t_bin"], df_bridge[metric], marker="o")
        plt.xticks(rotation=20)
        plt.ylabel(metric)
        plt.title(f"{metric} by t-bin")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTDIR / f"tbin_v2_{metric}.png", dpi=180)
        plt.show()


## 6. Rho and schedule scan on the same batch

In [ ]:

rho_batch, rho_args_eval = make_paired_batch(bp_args, n_samples=N_SAMPLES, kappa_override=bp_args.kappa, seed=3026)

rho_rows = []
for sched in RHO_SCHEDULES:
    for rho in RHO_LIST:
        mode = "none" if rho == 0 else "adaptive"
        row, _ = evaluate_on_batch(
            bp_model, bp_args, rho_batch, rho_args_eval,
            nfe=NFE_MAIN,
            rho_max=rho,
            projection_mode=mode,
            rho_sched=sched,
        )
        row["mode_name"] = f"{sched}_rho{rho}"
        rho_rows.append(row)

df_rho = pd.DataFrame(rho_rows)
df_rho.to_csv(OUTDIR / "rho_scan_metrics_v2.csv", index=False)
df_rho.head()


In [ ]:

def lineplot_rho(metric):
    plt.figure(figsize=(6,4))
    for sched, sub in df_rho.groupby("rho_sched"):
        sub = sub.sort_values("rho_max")
        plt.plot(sub["rho_max"], sub[metric], marker="o", label=sched)
    plt.xlabel("rho_max")
    plt.ylabel(metric)
    plt.title(f"Rho scan: {metric}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTDIR / f"rho_scan_v2_{metric}.png", dpi=180)
    plt.show()

for metric in ["endpoint_rmse", "trajectory_dist_to_Mt", "branch_acc", "excess_path_length", "path_tortuosity", "total_turning_angle_deg"]:
    lineplot_rho(metric)


## 7. Low-NFE scan on the same batch

In [ ]:

nfe_batch, nfe_args_eval = make_paired_batch(bp_args, n_samples=N_SAMPLES, kappa_override=bp_args.kappa, seed=4026)

nfe_rows = []
for nfe in NFE_LIST:
    row, _ = evaluate_on_batch(
        bp_model, bp_args, nfe_batch, nfe_args_eval,
        nfe=nfe,
        rho_max=0.0,
        projection_mode="none",
        rho_sched=RHO_SCHEDULE_MAIN,
    )
    row["mode_name"] = "BP-no-projection"
    nfe_rows.append(row)

    row, _ = evaluate_on_batch(
        bp_model, bp_args, nfe_batch, nfe_args_eval,
        nfe=nfe,
        rho_max=RHO_MAIN,
        projection_mode="adaptive",
        rho_sched=RHO_SCHEDULE_MAIN,
    )
    row["mode_name"] = "BP-projection"
    nfe_rows.append(row)

    if single_model is not None:
        row, _ = evaluate_on_batch(
            single_model, single_args, nfe_batch, nfe_args_eval,
            nfe=nfe,
            rho_max=0.0,
            projection_mode="none",
            rho_sched=RHO_SCHEDULE_MAIN,
        )
        row["mode_name"] = "Single-head"
        nfe_rows.append(row)

df_nfe = pd.DataFrame(nfe_rows)
df_nfe.to_csv(OUTDIR / "nfe_scan_metrics_v2.csv", index=False)
df_nfe.head()


In [ ]:

def lineplot_nfe(metric):
    plt.figure(figsize=(6,4))
    for name, sub in df_nfe.groupby("mode_name"):
        sub = sub.sort_values("nfe")
        plt.plot(sub["nfe"], sub[metric], marker="o", label=name)
    plt.xlabel("NFE")
    plt.ylabel(metric)
    plt.title(f"Low-NFE scan: {metric}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTDIR / f"nfe_scan_v2_{metric}.png", dpi=180)
    plt.show()

for metric in ["endpoint_rmse", "trajectory_dist_to_Mt", "branch_acc", "excess_path_length", "path_tortuosity", "total_turning_angle_deg"]:
    lineplot_nfe(metric)


## 8. Curvature stress test with paired batches for each kappa

In [ ]:

curv_rows = []

for kappa in KAPPA_LIST:
    batch_k, args_k = make_paired_batch(bp_args, n_samples=N_SAMPLES, kappa_override=kappa, seed=5026 + int(kappa * 1000))

    row, _ = evaluate_on_batch(
        bp_model, bp_args, batch_k, args_k,
        nfe=NFE_MAIN,
        rho_max=0.0,
        projection_mode="none",
        rho_sched=RHO_SCHEDULE_MAIN,
    )
    row["mode_name"] = "BP-no-projection"
    curv_rows.append(row)

    row, _ = evaluate_on_batch(
        bp_model, bp_args, batch_k, args_k,
        nfe=NFE_MAIN,
        rho_max=RHO_MAIN,
        projection_mode="adaptive",
        rho_sched=RHO_SCHEDULE_MAIN,
    )
    row["mode_name"] = "BP-projection"
    curv_rows.append(row)

    if single_model is not None:
        row, _ = evaluate_on_batch(
            single_model, single_args, batch_k, args_k,
            nfe=NFE_MAIN,
            rho_max=0.0,
            projection_mode="none",
            rho_sched=RHO_SCHEDULE_MAIN,
        )
        row["mode_name"] = "Single-head"
        curv_rows.append(row)

df_curv = pd.DataFrame(curv_rows)
df_curv.to_csv(OUTDIR / "curvature_stress_metrics_v2.csv", index=False)
df_curv.head()


In [ ]:

def lineplot_curv(metric):
    plt.figure(figsize=(6,4))
    for name, sub in df_curv.groupby("mode_name"):
        sub = sub.sort_values("kappa_eval")
        plt.plot(sub["kappa_eval"], sub[metric], marker="o", label=name)
    plt.xlabel("curvature kappa")
    plt.ylabel(metric)
    plt.title(f"Curvature stress: {metric}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTDIR / f"curvature_stress_v2_{metric}.png", dpi=180)
    plt.show()

for metric in ["endpoint_rmse", "trajectory_dist_to_Mt", "branch_acc", "excess_path_length", "path_tortuosity", "total_turning_angle_deg"]:
    lineplot_curv(metric)


## 9. Paired trajectory visualization

In [ ]:

def plot_curves(ax, args_eval, t_values=(0.0, 0.25, 0.5, 0.75, 1.0)):
    colors = {-1.0: "tab:blue", 1.0: "tab:orange"}
    for yval in [-1.0, 1.0]:
        for tv in t_values:
            curves = make_branch_curves(N_CURVE, DEVICE, kappa=args_eval.kappa, t_value=tv)[yval]
            mt = curves["mt"]
            if tv == 0.0:
                label = f"M0 y={int(yval)}"
                ls = "-"
                lw = 2.0
            elif tv == 1.0:
                label = f"M1 y={int(yval)}"
                ls = ":"
                lw = 2.0
            else:
                label = None
                ls = "--"
                lw = 1.0
            ax.plot(to_np(mt[:,0]), to_np(mt[:,1]), color=colors[yval], ls=ls, lw=lw, alpha=0.75, label=label)

def visualize_paired_trajectories(cache_a, cache_b, title_a, title_b, fname, n_show=32):
    traj_a = cache_a["traj"]
    traj_b = cache_b["traj"]
    args_eval = cache_a["args_eval"]

    idx = torch.linspace(0, traj_a.shape[1]-1, n_show).long().to(DEVICE)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.6), dpi=160)
    for ax, traj, title in [(axes[0], traj_a, title_a), (axes[1], traj_b, title_b)]:
        plot_curves(ax, args_eval)
        for ii in idx:
            tr = traj[:, ii]
            ax.plot(to_np(tr[:,0]), to_np(tr[:,1]), lw=0.8, alpha=0.55)
            ax.scatter(to_np(tr[-1:,0]), to_np(tr[-1:,1]), s=10)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=7)
        ax.set_title(title)
    plt.tight_layout()
    plt.savefig(OUTDIR / fname, dpi=180)
    plt.show()

visualize_paired_trajectories(
    cache_no, cache_proj,
    "BP-GI2SB without projection",
    f"BP-GI2SB with projection, rho={RHO_MAIN}",
    "paired_trajectory_no_vs_projection_v2.png",
)


## 10. Export all metrics to Excel

In [ ]:

summary_files = {
    "direct": OUTDIR / "direct_comparison_metrics_v2.csv",
    "bridge_tbin": OUTDIR / "bridge_diagnostic_by_tbin_v2.csv",
    "rho": OUTDIR / "rho_scan_metrics_v2.csv",
    "nfe": OUTDIR / "nfe_scan_metrics_v2.csv",
    "curvature": OUTDIR / "curvature_stress_metrics_v2.csv",
}

with pd.ExcelWriter(OUTDIR / "toy_curved_multibranch_v2_all_metrics.xlsx") as writer:
    for name, path in summary_files.items():
        if path.exists():
            pd.read_csv(path).to_excel(writer, sheet_name=name[:31], index=False)

print("Saved:", OUTDIR / "toy_curved_multibranch_v2_all_metrics.xlsx")
